In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
from tqdm import tqdm

# Basic settings
epochs = 2
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

# 5 different hyperparameter configs
hyperparameter_configs = [
    {'lr': 0.001, 'batch_size': 64,  'optimizer': 'Adam',   'model': 'BasicConvNet'},
    {'lr': 0.01,  'batch_size': 128, 'optimizer': 'SGD',    'model': 'ResNet18'},
    {'lr': 0.0005,'batch_size': 32,  'optimizer': 'Adam',   'model': 'MLP'},
    {'lr': 0.005, 'batch_size': 64,  'optimizer': 'SGD',    'model': 'BasicConvNet'},
    {'lr': 0.001, 'batch_size': 128, 'optimizer': 'Adam',   'model': 'ResNet18'},
]

# Model selector
def get_model(name):
    if name == 'BasicConvNet':
        return BasicConvNet()
    elif name == 'ResNet18':
        return ResNet18()
    elif name == 'MLP':
        return MLP()
    else:
        raise ValueError("Unknown model name")

# Optimizer selector
def get_optimizer(name, params, lr):
    if name == 'Adam':
        return optim.Adam(params, lr=lr)
    elif name == 'SGD':
        return optim.SGD(params, lr=lr, momentum=0.9)
    else:
        raise ValueError("Unknown optimizer")

# Training and evaluation loop
def train_and_evaluate(hparams, run_name):
    writer = SummaryWriter(comment=run_name)
    model = get_model(hparams['model']).to(device)
    optimizer = get_optimizer(hparams['optimizer'], model.parameters(), hparams['lr'])
    criterion = nn.CrossEntropyLoss()

    trainloader = DataLoader(trainset, batch_size=hparams['batch_size'], shuffle=True, num_workers=2)
    testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

    global_step = 0
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in tqdm(trainloader, desc=f"{run_name} | Epoch {epoch+1}/{epochs}"):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            writer.add_scalar('Training Loss', loss.item(), global_step)
            global_step += 1

        accuracy = 100. * correct / total
        writer.add_scalar('Training Accuracy', accuracy, epoch)

        # Evaluation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_acc = 100. * correct / total
        writer.add_scalar('Test Accuracy', test_acc, epoch)

    writer.close()

# Run multiple experiments
def run():
    for idx, hparams in enumerate(hyperparameter_configs):
        run_name = f"Run_{idx+1}_Model-{hparams['model']}_Opt-{hparams['optimizer']}_LR-{hparams['lr']}_BS-{hparams['batch_size']}"
        print(f"Starting {run_name}")
        train_and_evaluate(hparams, run_name)


100%|██████████| 170M/170M [00:11<00:00, 15.1MB/s]


In [ ]:
import torch.nn.functional as F

class BasicConvNet(nn.Module):
    def __init__(self):
        super(BasicConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


from torchvision.models import resnet18

class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        self.model = resnet18(weights=None)
        self.model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.model.maxpool = nn.Identity()
        self.model.fc = nn.Linear(512, 10)

    def forward(self, x):
        return self.model(x)
